# CFT Quickstart

This notebook demonstrates the basics of the CFT library:
creating agents, running two theories (CFT and GFT), and visualizing the results.

```bash
pip install -e .  # from the repo root
```

In [ ]:
import numpy as np
from cft import Agent, TheoryParameters, CFT, GFT
from cft.visualization import plot_groups, plot_affinity_matrix, plot_theory_comparison

## 1. Create Agents

Each agent has a feature vector representing opinions, personality traits, or other attributes.
We'll create 20 agents with 3 features, arranged in two loose clusters.

In [ ]:
rng = np.random.default_rng(42)

agents = []
# Cluster A: centered near (0, 0, 0)
for i in range(10):
    agents.append(Agent(id=i, features=rng.normal(0, 0.5, 3)))
# Cluster B: centered near (3, 3, 3)
for i in range(10):
    agents.append(Agent(id=10 + i, features=rng.normal(3, 0.5, 3)))

params = TheoryParameters(n_agents=20, n_features=3, random_seed=42)
print(f"Created {len(agents)} agents with {params.n_features} features each")

## 2. Run CFT (Consensus-Fracture Theory)

CFT forms groups where all pairwise affinities exceed a threshold.
Groups are crisp - you're in or you're out.

In [ ]:
cft = CFT(params, threshold=0.5)
cft.initialize_agents(agents)
cft_history = cft.run_simulation(t_max=5.0, dt=1.0)

cft_groups = cft.get_groups()
print(f"CFT found {len(cft_groups)} groups:")
for g in cft_groups:
    print(f"  Group {g.id}: {len(g.members)} members - {g.members}")

In [ ]:
fig = plot_groups(cft_groups, agents, title="CFT Groups")
fig

## 3. Run GFT (Gradient Field Theory)

GFT models agents as particles in a force field.
Agents attract similar neighbors and form clusters over time.

In [ ]:
gft = GFT(params, k=0.3, sigma=2.0)
gft.initialize_agents(agents)
gft_history = gft.run_simulation(t_max=10.0, dt=1.0)

gft_groups = gft.get_groups()
print(f"GFT found {len(gft_groups)} groups:")
for g in gft_groups:
    print(f"  Group {g.id}: {len(g.members)} members")

In [ ]:
fig = plot_groups(gft_groups, agents, title="GFT Groups")
fig

## 4. Compare Theories Side-by-Side

In [ ]:
fig = plot_theory_comparison(
    {"CFT": cft_history, "GFT": gft_history},
    agents,
)
fig

## 5. Visualize the Affinity Matrix

The affinity matrix shows how similar each pair of agents is.
Block structure indicates natural clusters.

In [ ]:
from cft import compute_affinity_matrix

aff = compute_affinity_matrix(agents)
fig = plot_affinity_matrix(aff, groups=cft_groups)
fig

## Next Steps

- See `theory_comparison.ipynb` for all 5 theories with scoring and parameter sweeps
- See `mirofish_demo.ipynb` for loading real social simulation data
- Try changing the `threshold`, `k`, and `sigma` parameters above and re-running